# Пилот: предсказываем пробег машины линейной моделью

**Вопрос ноутбука:** можно ли по объявлению на lalafo.kg (год, марка, цена, двигатель…) угадать,
сколько машина проехала?

**Зачем:** у трети машин пробег не указан. Модель подскажет «ожидаемый» пробег — и для пустых
объявлений, и чтобы заметить подозрительно маленький заявленный пробег.

**Как работаем** (по `linear_regression_checklist.md`):
1. смотрим на данные и на цель;
2. заранее решаем, какой результат считать хорошим;
3. строим простые модели-«планки» (бейзлайны);
4. проверяем гипотезы **по одной**: одно изменение → одно сравнение;
5. выбираем лучшую модель и **один раз** проверяем её на тесте;
6. разбираем, где она ошибается.

Это тот же расчёт, что в `mileage/pilot.py`, но по шагам и с объяснениями.
Итоги пилота на полных данных — в `mileage/pilot_report.md`.

### Что здесь нового (по сравнению с занятиями)

| понятие | одной фразой | раздел |
|---|---|---|
| кросс-валидация (K-fold) | 5 раз делим train на «учим / проверяем», чтобы оценка не зависела от одного удачного деления | 4 |
| `MissingIndicator` | столбец 0/1 «тут был пропуск» — модель узнаёт, что значение заполнили | 5 |
| `min_frequency` в `OneHotEncoder` | редкие категории склеиваются в одну «редкую» | 5 |
| логарифм цели | учим модель на `log(пробег)`, прогноз переводим обратно через `exp` | 6 |
| цель «пробег в год» | учим на `пробег / возраст`, прогноз умножаем на возраст | 6 |
| `RidgeCV` | Ridge, который сам подбирает `alpha` своей внутренней кросс-валидацией | 7 |
| out-of-fold прогноз | прогноз для каждой машины train от модели, которая её **не видела** | 9 |
| кривая обучения | как меняется ошибка, если давать модели больше данных | 9 |
| Q-Q plot | похожи ли остатки на нормальное распределение | 9 |
| VIF, расстояние Кука | дополнительно: насколько признаки дублируют друг друга и какие машины сильнее всего тянут модель | 12 |
| взаимодействие признаков | столбец `возраст × такси`: у такси пробег растёт с возрастом быстрее | 14 |
| сравнение по фолдам | разница двух моделей на **каждом** фолде — надёжнее, чем сравнивать средние | 14 |
| свой трансформер (`fit` / `transform`) | шаг `Pipeline`, который сам запоминает медианы цены на обучающих машинах | 15 |
| обрезка выбросов (clip) | значения дальше ±порога заменяем порогом, чтобы редкие ошибки не тянули вес | 15 |

## 0. Подготовка

**Что нужно:**
- файл `dataset/data/listings.parquet` — [Hugging Face](https://huggingface.co/datasets/aiacademy-kg/lalafo-kg-cars/blob/main/data/listings.parquet);
- библиотеки из `requirements.txt` (+ `joblib`, `jupyter`).

Код очистки данных живёт в `EDA/prep.py`, сборка таблицы для модели — в `mileage/build_dataset.py`.
Ноутбук их только **импортирует**, чтобы правила очистки были одни и те же везде.

In [ ]:
import os

# Матрицы маленькие: много потоков BLAS только мешают друг другу
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(_v, "4")

import json
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mt
import numpy as np
import pandas as pd
import scipy.stats as st
from IPython.display import display

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.impute import MissingIndicator, SimpleImputer
from sklearn.linear_model import LinearRegression, RidgeCV
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.model_selection import KFold, learning_curve
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Корень репозитория — чтобы ноутбук работал и из корня, и из папки mileage/
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "EDA" / "prep.py").exists())
sys.path[:0] = [str(ROOT / "EDA"), str(ROOT / "mileage")]
import prep            # очистка данных + стиль графиков
import build_dataset   # сборка таблицы для модели

warnings.filterwarnings("ignore", message=".*unknown categories.*")
warnings.filterwarnings("ignore", category=FutureWarning)
prep.set_style()
BLUE, ORANGE, GREEN = prep.SERIES[:3]
GREY = prep.NEUTRAL

RANDOM_STATE = 42


def km(v, _=None):
    """150000 -> '150k' для подписей осей."""
    return f"{v / 1000:,.0f}k".replace(",", " ")


def sp(v):
    """150000 -> '150 000' для текста."""
    return f"{v:,.0f}".replace(",", " ")

## 1. Данные

### 1.1 Таблица для модели

**Одной фразой:** одна строка = одна уникальная машина 2010–2024 года, у которой указан
правдоподобный пробег.

Сборка (`build_dataset.py`) делает так:
1. берёт все объявления и чистит их правилами из `prep.py` (обоснования — в `EDA/eda_report.md`);
2. выкидывает «Другие автомобили» (запчасти, спецтехника) и **повторы одной машины**;
3. оставляет годы 2010–2024 и пробег от 1 000 км и от 1 до 60 тыс. км в год
   (`5 000 000 км` или `10 км` у машины 2012 года — опечатка, а не реальный пробег);
4. **один раз** делит машины на train (80 %) и test (20 %) и запоминает это деление в таблице.

> ⚠️ **Режим «только listings».** У нас есть только `listings.parquet`, а пилот собирался ещё из
> `users`, `cities` и фото. Поэтому: области берутся из самих объявлений; повторы одной машины ищем
> только по совпадению «марка + модель + год + цена + продавец» (без сравнения фото — повторов
> находится меньше); «дилер» = продавец с ≥ 3 объявлениями. Цифры похожи на пилот, но не совпадают
> один в один. `LISTINGS_ONLY = False` вернёт полную сборку (нужны все файлы `dataset/data/`).

In [ ]:
LISTINGS_ONLY = True
DATA, SUMMARY = build_dataset.out_paths(LISTINGS_ONLY)

if not DATA.exists():  # первый запуск: собираем таблицу (несколько секунд)
    DATA.parent.mkdir(exist_ok=True)
    _data, _summary = build_dataset.build(LISTINGS_ONLY)
    _data.to_parquet(DATA, index=False)
    SUMMARY.write_text(json.dumps(_summary, ensure_ascii=False, indent=2))

df = pd.read_parquet(DATA)
print("таблица:", DATA.relative_to(ROOT), "| строк:", len(df))
funnel = pd.DataFrame(json.loads(SUMMARY.read_text())["funnel"])
funnel.columns = ["шаг отбора", "осталось машин"]
funnel

**Что здесь видно:** из 54 тыс. объявлений для модели остаётся ≈ 14 тыс. Больше всего строк уходит
на шаге «год 2010–2024» (старые машины нам не нужны) и «пробег указан» (у многих его просто нет).

### 1.2 Какие есть столбцы

In [ ]:
TARGET = "mileage_km"

NUM_BASE = ["age", "engine_l", "price_kgs", "image_count"]          # числа
CAT_BASE = ["brand", "body_type", "transmission", "drive", "steering_wheel", "condition",
            "tech_condition", "customs_cleared", "registration_country", "region",
            "seller", "currency", "is_vip"]                          # категории (текст)
BIN = ["fuel_petrol", "fuel_diesel", "fuel_lpg", "fuel_hybrid", "fuel_ev", "fuel_isna"]  # 0/1

# пропуски в категориях — настоящий np.nan (так их понимает SimpleImputer)
for c in CAT_BASE + ["brand_model"]:
    df[c] = df[c].astype(object).where(df[c].notna(), np.nan)

df[[TARGET] + NUM_BASE + ["brand", "brand_model", "seller", "fuel_lpg", "split"]].head()

- **Цель** `mileage_km` — пробег в км.
- **Числа:** `age` — возраст (2026 − год выпуска), `engine_l` — объём двигателя, `price_kgs` — цена
  в сомах, `image_count` — сколько фото в объявлении.
- **Категории:** марка, кузов, коробка, привод, руль, состояние, растаможка, страна учёта, область,
  тип продавца, валюта объявления, VIP-объявление.
- **Топливо** — 0/1-столбцы: продавец может отметить сразу «бензин» и «газ».
- `brand_model` («Toyota | Camry») используем позже, в гипотезе H5.
- `split` — train или test. **Деление не меняем**, иначе следующие версии модели нельзя будет
  сравнить с этой.

In [ ]:
pd.DataFrame({"пропусков, %": (df[NUM_BASE + CAT_BASE].isna().mean() * 100).round(1)}).T

**Что здесь видно:** пропуски есть в объёме двигателя, цене («договорная»), коробке, приводе и
некоторых категориях. Заполнять их будем внутри `Pipeline` (раздел 5) — **только по train**.

### 1.3 Train и test

**Одной фразой:** test — это машины, которые модель увидит **один раз в самом конце**.

Пока выбираем модель, test не трогаем совсем. Иначе мы невольно подгоним модель под него, и
оценка на тесте станет слишком оптимистичной.

In [ ]:
train = df[df.split == "train"].reset_index(drop=True)
test = df[df.split == "test"].reset_index(drop=True)
y_train = train[TARGET].to_numpy()
y_test = test[TARGET].to_numpy()

print(f"train: {len(train)} машин, test: {len(test)} машин")
pd.DataFrame({"train": train[TARGET].describe(), "test": test[TARGET].describe()}).T.round(0)

**Что здесь видно:** медиана, квартили и максимум в train и test почти одинаковые. Деление
стратифицировано по возрасту машины (в обеих частях одинаковая доля новых и старых), поэтому
test — честная «уменьшенная копия» train.

## 2. Смотрим на цель (только train)

### 2.1 Как распределён пробег

In [ ]:
km_per_year = train[TARGET] / train.age

fig, axs = plt.subplots(1, 3, figsize=(12, 3.4))
axs[0].hist(train[TARGET], bins=60, color=BLUE)
axs[0].set_title(f"Пробег, км (асимметрия {train[TARGET].skew():.2f})")
axs[0].xaxis.set_major_formatter(mt.FuncFormatter(km))
axs[1].hist(np.log(train[TARGET]), bins=60, color=BLUE)
axs[1].set_title(f"log(пробег) (асимметрия {np.log(train[TARGET]).skew():.2f})")
axs[2].hist(km_per_year, bins=60, color=BLUE)
axs[2].set_title(f"Пробег в год, км (асимметрия {km_per_year.skew():.2f})")
axs[2].xaxis.set_major_formatter(mt.FuncFormatter(km))
for ax in axs:
    ax.set_yticks([])
plt.tight_layout()
plt.show()

**Асимметрия** (skew) — насколько у распределения длинный хвост: 0 — симметрично, > 0 — хвост вправо.

**Что здесь видно:**
- обычный пробег — с хвостом вправо: немного машин с 400–900 тыс. км (часто такси);
- `log(пробег)` — хвост теперь **влево**: логарифм «пережал» и сделал перекос в другую сторону;
- у **пробега в год** хвост вправо немного короче, чем у обычного пробега.

Отсюда две гипотезы для раздела 6: учить модель на `log(пробег)` (H2) или на «пробеге в год» (H3).

### 2.2 От чего зависит пробег

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(13, 3.8))

# пробег по году выпуска: медиана и «коридор» из половины машин
g = train.groupby("year")[TARGET]
ax = axs[0]
ax.fill_between(g.median().index, g.quantile(.25), g.quantile(.75), color=BLUE, alpha=0.18, lw=0,
                label="середина: 50 % машин")
ax.plot(g.median().index, g.median(), color=BLUE, marker="o", ms=4, label="медиана")
ax.yaxis.set_major_formatter(mt.FuncFormatter(km))
ax.set_title("Пробег по году выпуска")
ax.legend(fontsize=8.5)

# пробег и цена
ax = axs[1]
ax.scatter(train.price_kgs, train[TARGET], s=3, alpha=0.15, color=BLUE)
ax.set_xscale("log")
ax.yaxis.set_major_formatter(mt.FuncFormatter(km))
ax.set_xlabel("цена, сом (лог. шкала)")
ax.set_title("Пробег и цена")

# годовой пробег по топливу
fuel_names = {"fuel_petrol": "бензин", "fuel_diesel": "дизель", "fuel_lpg": "газ",
              "fuel_hybrid": "гибрид", "fuel_ev": "электро"}
rows = sorted(((name, km_per_year[train[col] == 1].median(), int((train[col] == 1).sum()))
               for col, name in fuel_names.items()), key=lambda r: r[1])
ax = axs[2]
ax.barh([f"{r[0]} (n={sp(r[2])})" for r in rows], [r[1] for r in rows], color=BLUE, height=0.6)
ax.xaxis.set_major_formatter(mt.FuncFormatter(km))
ax.grid(axis="y", visible=False)
ax.set_title("Медианный пробег в год по топливу")

plt.tight_layout()
plt.show()

**Что здесь видно:**
- **возраст — главный признак:** чем старше машина, тем больше пробег. Но связь не совсем прямая:
  у старых машин рост замедляется (гипотеза H6 — квадрат возраста);
- **цена:** дешевле → больше пробег, но облако широкое. Цена — это ещё и марка, и класс машины;
- **топливо:** на газу ездят заметно больше в год (часто это такси).

In [ ]:
corr = train[[TARGET, "age", "log_age", "price_kgs", "log_price", "engine_l", "image_count"]].corr()[TARGET]
corr.drop(TARGET).round(2).to_frame("корреляция с пробегом")

**Что здесь видно:** сильнее всего с пробегом связан возраст (≈ 0,5). Цена связана отрицательно,
причём `log_price` чуть сильнее, чем просто цена, — это гипотеза H1.

## 3. Как будем оценивать — решаем ДО обучения

**Главная метрика — MAE в км:** «в среднем модель ошибается на X км». Её легко объяснить
человеку. Дополнительно смотрим RMSE (сильнее штрафует большие промахи) и $R^2$.

MAPE (ошибка в %) главной не берём: у новой машины пробег 5 000 км, и промах на 5 000 км — уже 100 %.

**Планки «хорошо / плохо» — фиксируем сейчас, чтобы потом не подгонять их под результат:**

| критерий | планка |
|---|---|
| модель лучше, чем «линейная регрессия только по возрасту» (B2) | на ≥ 10 % по MAE |
| ошибка на тесте близка к ошибке на кросс-валидации | в пределах ±5 % |
| цель на будущие итерации | MAE ≤ 35 тыс. км |

In [ ]:
BAR_VS_AGE_ONLY = 0.10
BAR_TEST_GAP = 0.05
BAR_LONG_TERM_KM = 35_000

## 4. 🆕 Кросс-валидация: как сравнивать модели, не трогая test

**Одной фразой:** делим train на 5 частей и 5 раз подряд учим модель на четырёх частях, а
проверяем на пятой. Каждая машина ровно один раз оказывается «на проверке».

**Зачем, если уже есть test?** Мы будем сравнивать ~10 моделей. Если каждую мерить на test, мы
выберем ту, которой **повезло** с тестом. Кросс-валидация (CV) живёт только внутри train.

**Почему не одна валидационная часть?** Ошибка сильно зависит от того, какие машины попали
на проверку (это было видно на занятии про метрики: $R^2$ от 0,74 до 0,81 при разных делениях).
Пять проверок дают **среднее** и **разброс** — по разбросу видно, значима ли разница между моделями.

In [ ]:
N_FOLDS = 5
cv = KFold(N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

fig, ax = plt.subplots(figsize=(8, 2.2))
part = len(train) / N_FOLDS
for k in range(N_FOLDS):
    ax.broken_barh([(0, len(train))], (k - 0.4, 0.8), color=GREY)
    ax.broken_barh([(k * part, part)], (k - 0.4, 0.8), color=ORANGE)
ax.set_yticks(range(N_FOLDS), [f"проход {k + 1}" for k in range(N_FOLDS)])
ax.invert_yaxis()
ax.set_xticks([])
ax.grid(False)
ax.set_title("5-fold CV: серым учим, оранжевым проверяем (машины перемешаны случайно)")
plt.show()

Ниже — **сердце ноутбука**, функция `evaluate`. Она делает CV «вручную», обычным циклом, чтобы
было видно каждый шаг. Внутри цикла:
1. берём 4 части train для обучения и 1 часть для проверки;
2. переводим цель в нужный вид (`raw` — как есть; `log` и `rate` — см. раздел 6);
3. учим модель и делаем прогноз;
4. переводим прогноз обратно **в км** и считаем MAE.

Ошибку на обучающих частях (`train MAE`) тоже считаем: если она сильно меньше ошибки на
проверке — модель переобучилась (зазубрила train).

In [ ]:
def to_target(y, age, target):
    """Километры -> то, на чём учится модель."""
    if target == "log":
        return np.log(y)
    if target == "rate":
        return y / age          # км в год
    return y


def from_target(t, age, target):
    """Прогноз модели -> обратно в километры."""
    if target == "log":
        return np.exp(t)
    if target == "rate":
        return t * age
    return t


def evaluate(exp, data=train, y=y_train):
    """5-fold CV одного эксперимента. Возвращает строку итогов и out-of-fold прогноз."""
    fold_mae, fold_rmse, fold_r2, fold_train_mae, alphas = [], [], [], [], []
    oof = np.zeros(len(data))   # прогноз для каждой машины от модели, которая её не видела
    for tr_idx, va_idx in cv.split(data):
        X_tr, X_va = data.iloc[tr_idx], data.iloc[va_idx]
        model = build_model(exp)
        model.fit(X_tr, to_target(y[tr_idx], X_tr.age.to_numpy(), exp["target"]))
        p_va = from_target(model.predict(X_va), X_va.age.to_numpy(), exp["target"])
        p_tr = from_target(model.predict(X_tr), X_tr.age.to_numpy(), exp["target"])
        oof[va_idx] = p_va
        fold_mae.append(mean_absolute_error(y[va_idx], p_va))
        fold_rmse.append(root_mean_squared_error(y[va_idx], p_va))
        fold_r2.append(r2_score(y[va_idx], p_va))
        fold_train_mae.append(mean_absolute_error(y[tr_idx], p_tr))
        if exp["model"] == "ridge":
            alphas.append(model.named_steps["model"].alpha_)
    row = {"id": exp["id"], "что меняем": exp["title"],
           "CV MAE": np.mean(fold_mae), "± (std по фолдам)": np.std(fold_mae),
           "train MAE": np.mean(fold_train_mae), "CV RMSE": np.mean(fold_rmse), "CV R²": np.mean(fold_r2),
           "признаков": n_features(exp)}
    if alphas:
        row["alpha"] = float(np.median(alphas))
    return row, oof

## 5. Подготовка признаков: `Pipeline`

**Одной фразой:** та же цепочка, что на занятиях: заполнить пропуски → привести числа к одному
масштабу → категории в столбцы 0/1 → регрессия.

Цепочка учится **внутри каждого прохода CV** только на обучающих частях — так медианы для
пропусков и средние для `StandardScaler` не «подсматривают» в проверочную часть.

Две новые детали:
- 🆕 **`MissingIndicator`** — для каждого числового столбца с пропусками добавляет столбец 0/1
  «здесь был пропуск». Заполнили цену медианой — но модель всё равно знает, что цена была
  «договорная», и может это учесть.
- 🆕 **`OneHotEncoder(min_frequency=30)`** — категории, у которых меньше 30 машин, склеиваются в одну
  `infrequent`. Для марки, у которой 3 объявления, вес всё равно не выучить надёжно.

**`drop="first"` или нет.** Для `LinearRegression` одну категорию в каждом столбце выкидываем
(«базовая»): иначе 0/1-столбцы одной группы в сумме дают 1 и дублируют свободный член — та самая
проблема «двух одинаковых столбцов» с занятия про Ridge. Ridge с ней справляется сам, поэтому ему
оставляем все категории — так каждый вес читается как «отклонение этой категории».

In [ ]:
MIN_FREQ = 30
ALPHAS = np.logspace(-3, 4, 71)   # из чего RidgeCV выбирает alpha: от 0.001 до 10 000


def make_prep(num, cat, binary, drop="first"):
    parts = []
    if num:
        parts.append(("num", Pipeline([("impute", SimpleImputer(strategy="median")),
                                       ("scale", StandardScaler())]), num))
        parts.append(("miss", MissingIndicator(features="missing-only"), num))
    if cat:
        parts.append(("cat", Pipeline([
            ("impute", SimpleImputer(strategy="constant", fill_value="unknown")),
            ("onehot", OneHotEncoder(drop=drop, min_frequency=MIN_FREQ,
                                     handle_unknown="infrequent_if_exist", sparse_output=False)),
        ]), cat))
    if binary:
        parts.append(("bin", "passthrough", binary))
    return ColumnTransformer(parts)

Посмотрим, что цепочка делает с пятью машинами (два числа и одна категория):

In [ ]:
demo_cols = ["engine_l", "price_kgs", "body_type"]
demo = make_prep(["engine_l", "price_kgs"], ["body_type"], [], drop=None).fit(train)
rows5 = train[train.price_kgs.isna()].head(2).index.tolist() + train.head(3).index.tolist()

print("ДО:")
display(train.loc[rows5, demo_cols])
print("ПОСЛЕ:")
pd.DataFrame(demo.transform(train.loc[rows5]), columns=demo.get_feature_names_out(), index=rows5).round(2)

**Что здесь видно:**
- `num__engine_l`, `num__price_kgs` — числа после `StandardScaler` (0 = среднее по train);
- `miss__price_kgs` = 1 у машин с договорной ценой — их цену заменили медианой;
- `cat__body_type_…` — по столбцу 0/1 на каждый кузов; редкие кузова попали в `…_infrequent_sklearn`.

In [ ]:
def build_model(exp):
    """Эксперимент (словарь) -> необученная модель."""
    if exp["model"] == "dummy_mean":
        return DummyRegressor(strategy="mean")
    if exp["model"] == "dummy_median":
        return DummyRegressor(strategy="median")
    if exp["model"] == "lr":
        reg, drop = LinearRegression(), "first"
    else:
        reg, drop = RidgeCV(alphas=ALPHAS), None
    return Pipeline([("prep", make_prep(exp["num"], exp["cat"], exp["bin"], drop=drop)),
                     ("model", reg)])


def n_features(exp):
    """Сколько столбцов получает регрессия после подготовки."""
    if exp["model"].startswith("dummy"):
        return 0
    prep_ = build_model(exp).named_steps["prep"].fit(train)
    return len(prep_.get_feature_names_out())

## 6. Эксперименты: планки и гипотезы

Каждый эксперимент — словарь: какая модель, какая цель, какие признаки. Одно изменение
относительно предыдущего, чтобы было понятно, **что именно** помогло.

### 6.1 Бейзлайны — модели, которые надо обогнать

- **B0 — среднее.** Всем машинам предсказываем средний пробег. Модель, которая ничего не знает.
- **B1 — медиана.** То же, но медиана: для MAE это лучшая константа.
- **B2 — LR только по возрасту.** `пробег ≈ a + b · возраст`. Самый сильный признак в одиночку.

In [ ]:
results, oofs = [], {}


def run(exp):
    row, oof = evaluate(exp)
    results.append(row)
    oofs[exp["id"]] = oof
    print(f'{exp["id"]:<3} {exp["title"]:<40} CV MAE {sp(row["CV MAE"]):>7} ± {sp(row["± (std по фолдам)"]):>5} км'
          f' | train MAE {sp(row["train MAE"]):>7} | R² {row["CV R²"]:.3f}')
    return exp


NO_FEATURES = dict(num=[], cat=[], bin=[])
B0 = run(dict(id="B0", title="среднее", model="dummy_mean", target="raw", **NO_FEATURES))
B1 = run(dict(id="B1", title="медиана", model="dummy_median", target="raw", **NO_FEATURES))
B2 = run(dict(id="B2", title="LR: только возраст", model="lr", target="raw", num=["age"], cat=[], bin=[]))

**Что здесь видно:** константа ошибается в среднем на ≈ 60 тыс. км. Один только возраст снижает
ошибку до ≈ 52 тыс. км. **Планка для нашей модели — B2 минус 10 %, то есть ≈ 47 тыс. км.**

### 6.2 E1 — все базовые признаки

**Гипотеза:** характеристики машины, цена и тип продавца добавляют информацию к возрасту.

In [ ]:
BASE = dict(num=NUM_BASE, cat=CAT_BASE, bin=BIN)
E1 = run(dict(id="E1", title="LR: все базовые признаки", model="lr", target="raw", **BASE))

**Что здесь видно:** ошибка упала с ≈ 52 до ≈ 46 тыс. км — планка уже пройдена. `train MAE` почти
равен `CV MAE` — модель **не** переобучается.

### 6.3 E2 — H1: логарифм цены

**Гипотеза:** важна не разница в цене в сомах, а **во сколько раз** машина дешевле. Машина за
5 000 $ и за 10 000 $ отличаются сильнее, чем за 50 000 $ и 55 000 $. Логарифм превращает
«во сколько раз» в «на сколько» — и связь становится линейнее.

In [ ]:
LOGP = dict(num=["age", "engine_l", "log_price", "image_count"], cat=CAT_BASE, bin=BIN)
E2 = run(dict(id="E2", title="E1, но log(цены)", model="lr", target="raw", **LOGP))

**Что здесь видно:** разница с E1 в несколько километров — меньше, чем разброс между фолдами (±).
Гипотеза не подтвердилась, но и не навредила; оставляем `log_price`, потому что с ним проще
читать коэффициенты («цена в 2 раза выше → …»).

### 6.4 🆕 E3 — H2: учим на log(пробега)

**Гипотеза:** `пробег = возраст^b × интенсивность` — это **произведение**. После логарифма оно
становится суммой: `log(пробег) = b·log(возраст) + log(интенсивность)` — то, что умеет линейная модель.

**Как это работает:** модель учим на `log(пробег)`, а её прогноз переводим обратно: `exp(прогноз)`.
MAE считаем, как и раньше, **в км** — иначе модели нельзя сравнить.

### 🆕 E4 — H3: учим на «пробег в год»

**Гипотеза:** годовой пробег распределён ровнее (раздел 2.1). Модель предсказывает «сколько км в
год ездит такая машина», а прогноз в км = `ставка × возраст`.

Обе цели уже «зашиты» в функции `to_target` / `from_target` из раздела 4.

In [ ]:
LOGAGE = dict(num=["log_age", "engine_l", "log_price", "image_count"], cat=CAT_BASE, bin=BIN)
E3 = run(dict(id="E3", title="E2 + цель log(пробега), log(возраста)", model="lr", target="log", **LOGAGE))
E4 = run(dict(id="E4", title="E2 + цель «пробег в год»", model="lr", target="rate", **LOGP))

**Что здесь видно:** обе «умные» цели ошибаются **больше**, чем E2 (на ≈ 600–700 км). Почему так:
- модель на `log` минимизирует ошибку в процентах, а мы меряем в км — её «устраивает» большой промах
  в км у машины с большим пробегом;
- у «пробега в год» ошибка ставки умножается на возраст: промах 3 тыс. км/год у 15-летней машины
  превращается в 45 тыс. км.

Вывод: дальше идём с обычной целью — то есть с формой E2. Код выбирает её автоматически, по CV:

In [ ]:
best_form = min([E2, E3, E4], key=lambda e: next(r["CV MAE"] for r in results if r["id"] == e["id"]))
print("лучшая форма среди E2–E4:", best_form["id"], "—", best_form["title"])

## 7. Ridge и новые признаки

### 7.1 🆕 E5 — H4: Ridge с подбором alpha (`RidgeCV`)

**Гипотеза:** у нас ~90 столбцов, большинство — 0/1 по категориям, и многие из них похожи
(например, «правый руль» почти всегда «учёт в Японии»). Ridge «берёт веса поменьше» и стабилизирует их.

**`RidgeCV`** — Ridge, который сам перебирает `alpha` из списка `ALPHAS` и выбирает лучший своей
**внутренней** кросс-валидацией (на обучающих частях текущего прохода). То же, что мы делали
вручную на занятии: перебор `alpha` → ошибка на проверке → лучший.

In [ ]:
E5 = run(dict(best_form, id="E5", title=f"{best_form['id']} + Ridge (alpha по CV)", model="ridge"))
print("выбранный alpha (медиана по фолдам):", round(results[-1]["alpha"], 2))

**Что здесь видно:** выигрыш маленький — признаков немного, а машин много (≈ 100 машин на столбец),
поэтому обычная регрессия и так не «прыгала». Ridge нам пригодится в следующем шаге, когда столбцов
станет больше.

### 7.2 E6 — H5: конкретная модель машины

**Гипотеза:** марки мало: Toyota бывает и Camry (часто такси), и Land Cruiser. Добавляем
`brand_model` — «марка | модель», если у неё ≥ 30 объявлений (остальные — в `infrequent`).

In [ ]:
E6 = run(dict(E5, id="E6", title="E5 + марка×модель (≥ 30 объявлений)", cat=CAT_BASE + ["brand_model"]))

**Что здесь видно:** самый большой шаг после E1 — минус ≈ 1,3 тыс. км. Конкретная модель правда
говорит о том, как на машине ездят. Столбцов стало ≈ 170 — здесь Ridge уже нужен.

### 7.3 E7 — H6: квадрат возраста

**Гипотеза:** пробег растёт с возрастом не по прямой (раздел 2.2: у старых машин рост замедляется).
Добавляем `(возраст − 7)²`. Вычитаем 7 (типичный возраст), чтобы новый столбец меньше дублировал
сам `age` — иначе `age` и `age²` почти одинаковые.

In [ ]:
AGE_CENTER = 7.0
for d in (df, train, test):
    d["age_c2"] = (d.age - AGE_CENTER) ** 2
E7 = run(dict(E6, id="E7", title="E6 + квадрат возраста", num=E6["num"] + ["age_c2"]))

## 8. Итог экспериментов

In [ ]:
res = pd.DataFrame(results)
display(res.round({"CV MAE": 0, "± (std по фолдам)": 0, "train MAE": 0, "CV RMSE": 0, "CV R²": 3, "alpha": 2}))

ALL = {e["id"]: e for e in [B0, B1, B2, E1, E2, E3, E4, E5, E6, E7]}
best_id = res[res.id.str.startswith("E")].sort_values("CV MAE").id.iloc[0]
best = ALL[best_id]
r_best, r_b2 = res.set_index("id").loc[best_id], res.set_index("id").loc["B2"]
print(f"лучшая модель: {best_id} ({best['title']})")
print(f"лучше B2 на {1 - r_best['CV MAE'] / r_b2['CV MAE']:.1%} (планка {BAR_VS_AGE_ONLY:.0%})")

In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 4.8))
order = res.iloc[::-1]
colors = [GREY if i.startswith("B") else (ORANGE if i == best_id else BLUE) for i in order.id]
ax.barh(range(len(order)), order["CV MAE"], xerr=order["± (std по фолдам)"], color=colors, height=0.6)
ax.set_yticks(range(len(order)), [f"{i}  {t}" for i, t in zip(order.id, order["что меняем"])], fontsize=8.5)
for i, v in enumerate(order["CV MAE"]):
    ax.text(v + 1500, i, f"{v / 1000:.1f}k", va="center", fontsize=8.5)
ax.axvline(r_b2["CV MAE"] * (1 - BAR_VS_AGE_ONLY), color="k", lw=1, ls="--")
ax.text(r_b2["CV MAE"] * (1 - BAR_VS_AGE_ONLY), -0.9, " планка: B2 − 10 %", fontsize=8)
ax.xaxis.set_major_formatter(mt.FuncFormatter(km))
ax.set_xlabel("MAE на кросс-валидации, км (усы — разброс между фолдами)")
ax.grid(axis="y", visible=False)
ax.set_title("Эксперименты (серые — бейзлайны, оранжевая — выбранная модель)")
plt.show()

**Что здесь видно:** больше всего дали «все признаки» (E1) и «марка×модель» (E6). Логарифмы
и «пробег в год» не помогли. Все E-модели проходят планку, лучшая — E7.

## 9. Где ошибается выбранная модель

### 🆕 9.1 Out-of-fold прогноз

**Одной фразой:** при кросс-валидации каждая машина train один раз была «на проверке». Прогноз,
сделанный в тот момент, — **out-of-fold** (OOF). Это честный прогноз для всех 11 тыс. машин train,
и его можно разбирать, не трогая test. Функция `evaluate` уже сохранила его в `oofs`.

In [ ]:
oof = oofs[best_id]
resid = y_train - oof                 # остаток: + модель занизила, − завысила
print(f"OOF: MAE {sp(mean_absolute_error(y_train, oof))} км, R² {r2_score(y_train, oof):.3f}")
print("отрицательных прогнозов:", int((oof < 0).sum()), "| меньше 10 000 км:", int((oof < 10_000).sum()))

fig, axs = plt.subplots(1, 2, figsize=(11, 4.5))
ax = axs[0]
ax.scatter(y_train, oof, s=3, alpha=0.15, color=BLUE)
lim = [0, np.percentile(y_train, 99.5)]
ax.plot(lim, lim, color=ORANGE, ls="--", lw=1.5, label="идеальный прогноз")
ax.set_xlim(lim); ax.set_ylim(lim)
ax.xaxis.set_major_formatter(mt.FuncFormatter(km)); ax.yaxis.set_major_formatter(mt.FuncFormatter(km))
ax.set_xlabel("настоящий пробег"); ax.set_ylabel("прогноз (OOF)")
ax.set_title("Факт и прогноз"); ax.legend(fontsize=8.5)

ax = axs[1]
ax.scatter(oof, resid, s=3, alpha=0.15, color=BLUE)
ax.axhline(0, color="k", lw=0.8)
ax.xaxis.set_major_formatter(mt.FuncFormatter(km)); ax.yaxis.set_major_formatter(mt.FuncFormatter(km))
ax.set_xlabel("прогноз"); ax.set_ylabel("остаток = факт − прогноз")
ax.set_title("Остатки")
plt.tight_layout()
plt.show()

**Что здесь видно:**
- облако «положено» положе оранжевой линии: машины с маленьким пробегом модель **завышает**,
  с большим — **занижает**. Прогнозы стягиваются к середине — типичный признак **недообучения**:
  признакам не хватает информации, чтобы отличить такси от машины, которая стоит в гараже;
- остатки — «веер»: чем больше прогноз, тем шире разброс. Для старых машин ошибка в км больше —
  это естественно, пробег у них и сам больше.

### 🆕 9.2 Q-Q plot: нормальны ли остатки

**Одной фразой:** сортируем остатки и сравниваем с тем, какими они были бы у нормального
распределения. Если точки лежат на прямой — остатки нормальные.

Для прогноза нормальность **не нужна** — MAE и так честная. Она нужна, если хотим доверять
p-value и доверительным интервалам у коэффициентов (как в `statsmodels`).

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(11, 3.8))
axs[0].hist(resid, bins=80, color=BLUE)
axs[0].xaxis.set_major_formatter(mt.FuncFormatter(km))
axs[0].set_yticks([])
axs[0].set_title(f"Распределение остатков (асимметрия {st.skew(resid):.2f})")
(osm, osr), (slope, intercept, _) = st.probplot(resid, dist="norm")
axs[1].scatter(osm, osr, s=4, color=BLUE, alpha=0.5)
axs[1].plot(osm, slope * osm + intercept, color=ORANGE, lw=1.5)
axs[1].yaxis.set_major_formatter(mt.FuncFormatter(km))
axs[1].set_xlabel("квантили нормального распределения")
axs[1].set_ylabel("остатки")
axs[1].set_title("Q-Q plot")
plt.tight_layout()
plt.show()

**Что здесь видно:** в середине точки на прямой, а справа загибаются вверх: есть «длинный хвост»
машин, у которых пробег намного больше прогноза (такси, коммерческие машины). p-value
коэффициентов поэтому стоит воспринимать осторожно.

### 9.3 Ошибка по группам машин

In [ ]:
seg = train.assign(err=np.abs(resid), res=resid,
                   age_band=pd.cut(train.age, [1, 4, 7, 11, 16],
                                   labels=["2022–24", "2019–21", "2015–18", "2010–14"]))


def by(col):
    t = seg.groupby(col, observed=True).agg(машин=("err", "size"), MAE=("err", "mean"),
                                            смещение=("res", "mean"), медиана_пробега=(TARGET, "median"))
    t["MAE в % от медианы"] = (t.MAE / t.медиана_пробега * 100).round(0)
    return t.round(0)


print("По году выпуска:"); display(by("age_band"))
print("По типу продавца:"); display(by("seller"))
top_brands = seg.brand.value_counts().head(10).index
print("По маркам (10 самых частых):")
by("brand").loc[top_brands].sort_values("MAE")

**Смещение** — средний остаток: > 0 — модель в среднем **занижает** пробег в этой группе.

**Что здесь видно:**
- в км модель ошибается сильнее всего на старых машинах, а **в процентах** — на новых: у машин
  2022–24 годов медианный пробег ≈ 63 тыс. км, и средний промах ≈ 31 тыс. — это уже половина;
- смещения по группам небольшие: систематически ни одну группу модель не «забывает».

### 9.4 Самые большие промахи

In [ ]:
seg.assign(прогноз=oof.round(0)).nlargest(10, "err")[
    ["brand_model", "year", TARGET, "прогноз", "price_kgs", "seller", "condition", "mileage_rule"]]

**Что здесь видно:** самые большие ошибки — машины с огромным заявленным пробегом (часто
круглые числа или заглушки) и машины, у которых пробег пришлось чинить (`mileage_rule = x1000`:
продавец написал «300» вместо «300 000»). Часть ошибки — **шум в самой цели**, его моделью не убрать.

### 🆕 9.5 Кривая обучения: поможет ли больше данных?

**Одной фразой:** учим модель на 10 %, 20 %, …, 100 % train и смотрим ошибку на обучении и на CV.

Как читать:
- **кривые далеко друг от друга** (train ≪ CV) — переобучение: поможет больше данных или регуляризация;
- **кривые рядом и обе высоко** — недообучение: данных хватает, не хватает **признаков**.

In [ ]:
sizes, tr_scores, va_scores = learning_curve(
    build_model(best), train, y_train, cv=cv, train_sizes=np.linspace(0.1, 1, 8),
    scoring="neg_mean_absolute_error", shuffle=True, random_state=RANDOM_STATE)
lc = pd.DataFrame({"машин в обучении": sizes, "train MAE": -tr_scores.mean(1), "CV MAE": -va_scores.mean(1)})
display(lc.round(0))

fig, ax = plt.subplots(figsize=(8, 3.6))
ax.plot(lc["машин в обучении"], lc["train MAE"], color=GREY, marker="o", ms=4, label="train")
ax.plot(lc["машин в обучении"], lc["CV MAE"], color=BLUE, marker="o", ms=4, label="CV")
ax.yaxis.set_major_formatter(mt.FuncFormatter(lambda v, _: f"{v / 1000:.1f}k"))
ax.set_xlabel("сколько машин в обучении"); ax.set_ylabel("MAE, км")
ax.set_title(f"Кривая обучения ({best_id})"); ax.legend()
plt.show()

**Что здесь видно:** кривые сошлись почти вплотную, и CV-кривая в конце почти не снижается.
Это **недообучение**: удвоить данные — почти бесполезно, нужны новые признаки (например,
«такси-модель × газ», цена относительно таких же машин того же года).

## 10. Финальная модель и её коэффициенты

Учим выбранную модель **на всём train** — это та модель, которую мы проверим на тесте.

Как читать коэффициенты:
- **числа** прошли `StandardScaler`, поэтому вес — эффект изменения на 1 стандартное отклонение.
  Делим на `scale_` и получаем эффект на **1 единицу** (1 год, 1 литр);
- **категории** (Ridge, без базовой) — насколько эта категория выше/ниже средней.

In [ ]:
final = build_model(best).fit(train, y_train)
names = final.named_steps["prep"].get_feature_names_out()
coef = pd.Series(final.named_steps["model"].coef_, index=names)
print(f"признаков: {len(names)}, alpha = {final.named_steps['model'].alpha_:.2f}")

scaler = final.named_steps["prep"].named_transformers_["num"].named_steps["scale"]
per_unit = pd.DataFrame({
    "на 1 SD, км": coef[[f"num__{c}" for c in best["num"]]].to_numpy(),
    "1 SD =": scaler.scale_,
    "на 1 единицу, км": coef[[f"num__{c}" for c in best["num"]]].to_numpy() / scaler.scale_,
}, index=best["num"])
per_unit.round(2)

**Как читать (на примере `age`):** «+1 год возраста → примерно +12 тыс. км пробега» (плюс поправка
от квадрата возраста `age_c2`). `log_price`: если цена выше в $e ≈ 2,7$ раза, пробег ниже на
≈ 24 тыс. км (при тех же остальных признаках).

In [ ]:
top = coef.reindex(coef.abs().sort_values(ascending=False).index).head(20)[::-1]
labels = [n.split("__", 1)[1] for n in top.index]

fig, ax = plt.subplots(figsize=(9, 6.5))
ax.barh(labels, top, color=[BLUE if v > 0 else ORANGE for v in top], height=0.65)
ax.axvline(0, color="k", lw=0.8)
ax.grid(axis="y", visible=False)
ax.xaxis.set_major_formatter(mt.FuncFormatter(lambda v, _: f"{v / 1000:+.0f}k" if v else "0"))
ax.set_xlabel("эффект на пробег, км (числа — на 1 SD, категории — отклонение от средней)")
ax.set_title("20 самых больших коэффициентов (синий — пробег выше, оранжевый — ниже)")
plt.show()

**Что здесь видно:**
- больше всего весов — у **конкретных моделей** (гипотеза H5). Сильно «плюс»: Volkswagen Polo,
  Hyundai Solaris, Kia Rio — типичные такси (≈ +80…100 тыс. км к таким же машинам), Mercedes Sprinter —
  коммерческий бус. Газ (`fuel_lpg`) добавляет ≈ +37 тыс. км;
- сильный «минус» у ВАЗ 2107 (≈ −113 тыс. км). Скорее всего, это не «мало ездят», а **пятизначный
  одометр**: после 99 999 он начинает с нуля, и продавец пишет то, что видит;
- осторожно со сравнением: вес категории — эффект «машина этой модели против средней», а вес числа —
  эффект на 1 SD. Возраст (+37 тыс. на 1 SD ≈ 3 года) остаётся главным признаком, потому что он
  меняется у **каждой** машины, а модель Polo — у немногих.

## 11. Test — один раз

Модель выбрана, гипотезы проверены. Теперь **единственный раз** считаем ошибку на test.
Для сравнения на тех же машинах — бейзлайны B0 и B2 (их тоже учим на всём train).

In [ ]:
def test_metrics(model, target="raw"):
    p = from_target(model.predict(test), test.age.to_numpy(), target)
    return {"MAE": mean_absolute_error(y_test, p), "RMSE": root_mean_squared_error(y_test, p),
            "R²": r2_score(y_test, p)}, p


m_test, pred_test = test_metrics(final)
table = pd.DataFrame({best_id: m_test,
                      "B2 (возраст)": test_metrics(build_model(B2).fit(train, y_train))[0],
                      "B0 (среднее)": test_metrics(build_model(B0).fit(train, y_train))[0]}).T
display(table.round({"MAE": 0, "RMSE": 0, "R²": 3}))

gap = m_test["MAE"] / r_best["CV MAE"] - 1
print(f"test MAE {sp(m_test['MAE'])} км vs CV MAE {sp(r_best['CV MAE'])} км -> разница {gap:+.1%} "
      f"(планка ±{BAR_TEST_GAP:.0%}: {'✅' if abs(gap) <= BAR_TEST_GAP else '❌'})")
print(f"лучше B2 на тесте на {1 - m_test['MAE'] / table.loc['B2 (возраст)', 'MAE']:.1%}")
print(f"цель на будущее MAE ≤ {sp(BAR_LONG_TERM_KM)} км: {'✅' if m_test['MAE'] <= BAR_LONG_TERM_KM else '❌ пока нет'}")

**Что здесь видно:** ошибка на тесте почти такая же, как на кросс-валидации. Значит, CV
не обманывала нас, утечки и переобучения нет. Обе планки пилота пройдены, долгосрочная цель — нет.

## 12. Дополнительно: VIF и расстояние Кука

Этот раздел можно пропустить при первом чтении. Он проверяет два допущения линейной регрессии
с помощью `statsmodels`. Для этого нужна обычная (без Ridge) регрессия с базовыми категориями.

### 🆕 VIF — насколько признак дублирует остальные

**Одной фразой:** для каждого признака строим регрессию «этот признак по всем остальным».
Если остальные предсказывают его почти идеально ($R^2 → 1$), VIF $= 1 / (1 - R^2)$ становится большим.

- VIF ≈ 1 — признак независим;
- VIF > 10 — признак почти копия других: его вес в обычной регрессии «прыгает» (как в примере
  с двумя площадями на занятии про Ridge).

In [ ]:
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import OLSInfluence, variance_inflation_factor

diag_prep = make_prep(best["num"], best["cat"], best["bin"], drop="first").fit(train)
X_diag = diag_prep.transform(train).astype(float)
diag_names = diag_prep.get_feature_names_out()
keep = X_diag.std(0) > 0                       # столбцы-константы убираем
X_diag, diag_names = X_diag[:, keep], diag_names[keep]
X_const = sm.add_constant(X_diag)

vif = pd.Series([variance_inflation_factor(X_const, i) for i in range(1, X_const.shape[1])],
                index=[n.split("__", 1)[1] for n in diag_names]).sort_values(ascending=False)
print(f"VIF > 10: {(vif > 10).sum()} признаков из {len(vif)}")
vif.head(10).round(1).to_frame("VIF")

**Что здесь видно:** самые большие VIF — у **марок**. Марка почти полностью «записана» в столбцах
её моделей: если у машины `Hyundai | Solaris` = 1, то и `brand_Hyundai` = 1. Это причина, почему мы взяли
**Ridge**: он делает веса таких пар устойчивыми. На прогноз VIF не влияет — только на то,
насколько можно верить отдельным весам.

### 🆕 Расстояние Кука — какие машины сильнее всего тянут модель

**Одной фразой:** насколько изменятся **все** прогнозы, если выкинуть одну машину и переучить модель.
Обычный порог «подозрительности» — $4 / n$.

In [ ]:
ols = sm.OLS(y_train, X_const).fit()
cook = OLSInfluence(ols).cooks_distance[0]
thr = 4 / len(y_train)
print(f"машин с расстоянием Кука > 4/n: {(cook > thr).sum()} ({(cook > thr).mean():.1%}), максимум {cook.max():.3f}")
train.assign(cook=cook).nlargest(5, "cook")[["brand_model", "year", TARGET, "price_kgs", "cook"]]

**Что здесь видно:** «влиятельных» машин ≈ 5 %, но максимум далёк от 1 — ни одна машина в
одиночку не ломает модель. Удалять их **не** будем: это не ошибки, а редкие реальные машины
(большой пробег, редкая модель).

## 13. Итоги

1. **Лучшая модель — E7:** Ridge, все базовые признаки + «марка×модель» + квадрат возраста.
2. Она **проходит обе планки пилота:** лучше «только возраст» более чем на 10 %, тест совпал с CV.
3. **Не помогли:** логарифм цели и «пробег в год» — при метрике в км они проигрывают обычной цели.
4. **Диагноз — недообучение:** train ≈ CV, кривая обучения выровнялась. Дальше нужны новые признаки,
   а не больше данных и не другая регуляризация.
5. Часть ошибки — шум в самой цели (круглые числа, заглушки, сброс одометра).

### Что попробовать дальше
- взаимодействия: «такси-модель × газ», «возраст × тип кузова»;
- цена **относительно** таких же машин того же года (дешевле своих — возможно, больше пробег);
- почистить цель: заглушки вида 55 555, 99 999 и сброс пятизначного одометра у старых ВАЗ.

Каждую идею — отдельным экспериментом, сравнивая CV MAE с E7 (≈ 44 тыс. км) на **том же** сплите.

### Глоссарий

| термин | значение |
|---|---|
| бейзлайн | простая модель-планка, которую обязана обогнать настоящая |
| фолд | одна из 5 частей train в кросс-валидации |
| CV MAE | средняя MAE по 5 проверочным фолдам |
| OOF-прогноз | прогноз для машины от модели, которая её не видела при обучении |
| смещение | средний остаток в группе: > 0 — модель занижает |
| недообучение | модель слишком простая: ошибка высокая и на train, и на CV |
| переобучение | модель зазубрила train: на train ошибка намного ниже, чем на CV |
| VIF | во сколько раз «раздувается» неуверенность в весе из-за похожих признаков |
| расстояние Кука | насколько сдвинутся прогнозы, если убрать одну машину |

---
# Итерация 2

Пилот закончен, его числа зафиксированы выше. Дальше — новые эксперименты по одному.
Каждый сравниваем с **E7** на тех же фолдах и, если он лучше, один раз проверяем на том же тесте.

## 14. E9 — такси-модели и взаимодействия с возрастом

**Идея.** У такси и машин на газу пробег намного больше. Модель уже знает конкретные модели
(`brand_model`) и газ (`fuel_lpg`), но в E7 каждый такой признак — это **сдвиг**: «+N км ко всем
машинам этой модели, независимо от возраста». А такси за каждый год проезжает больше, поэтому
разница должна **расти с возрастом**.

### Какие модели считаем «такси»

Hyundai Sonata, Kia K5, Kia K7, Hyundai Grandeur (корейские седаны, 43–63 % машин на газу) и
Volkswagen Polo, Kia Rio, Hyundai Solaris (бюджетные седаны, типичные такси в СНГ).

> ⚠️ Список составлен по разбору данных (доля газа, годовой пробег, упоминания в объявлениях),
> при котором смотрели все машины, включая test. Это лёгкое «подглядывание». Чтобы его не было,
> список такси-моделей надо выбирать только по train — хорошая задача для следующей итерации.

In [ ]:
TAXI_MODELS = ["Hyundai | Sonata", "Kia | K5", "Kia | K7", "Hyundai | Grandeur",
               "Volkswagen | Polo", "Kia | Rio", "Hyundai | Solaris"]

for d in (df, train, test):
    d["taxi_model"] = d.brand_model.isin(TAXI_MODELS).astype(float)
    d["age_x_taxi"] = d.age * d.taxi_model    # 0 у не-такси, возраст — у такси
    d["age_x_lpg"] = d.age * d.fuel_lpg       # 0 без газа, возраст — на газу

print(f"такси-моделей в train: {train.taxi_model.mean():.1%}, машин на газу: {train.fuel_lpg.mean():.1%}")
train[["brand_model", "age", "fuel_lpg", "taxi_model", "age_x_taxi", "age_x_lpg"]].head(6)

### 🆕 Что такое взаимодействие

**Одной фразой:** произведение двух признаков даёт модели **свой наклон** для группы.

Без взаимодействия модель — это две параллельные прямые:

$$\text{пробег} = a + b \cdot \text{возраст} + c \cdot \text{такси}$$

У такси прямая просто сдвинута на $c$, наклон $b$ общий.

С взаимодействием:

$$\text{пробег} = a + b \cdot \text{возраст} + c \cdot \text{такси} + d \cdot (\text{возраст} \times \text{такси})$$

У не-такси столбец `возраст × такси` = 0, и наклон по-прежнему $b$. У такси он равен возрасту, и
наклон становится $b + d$. Если $d > 0$ — такси набирают пробег быстрее.

Проверяем по одному изменению:
- **E9a** — только флаг «такси-модель»;
- **E9b** — E9a + `возраст × такси`;
- **E9c** — E7 + `возраст × газ` (без такси-флага);
- **E9** — всё вместе.

In [ ]:
E9a = run(dict(E7, id="E9a", title="E7 + флаг такси-модели", bin=E7["bin"] + ["taxi_model"]))
E9b = run(dict(E9a, id="E9b", title="E9a + возраст × такси", num=E7["num"] + ["age_x_taxi"]))
E9c = run(dict(E7, id="E9c", title="E7 + возраст × газ", num=E7["num"] + ["age_x_lpg"]))
E9 = run(dict(E9a, id="E9", title="E9a + возраст×такси + возраст×газ", num=E7["num"] + ["age_x_taxi", "age_x_lpg"]))

### 🆕 Сравнение по фолдам

Разница средних небольшая, а разброс по фолдам ±250 км. Как понять, что улучшение настоящее?

Сравним модели **на каждом фолде отдельно**: фолды у всех экспериментов одинаковые
(`random_state=42`), поэтому на фолде 1 обе модели проверяются на одних и тех же машинах.
Если новая модель лучше на всех 5 фолдах, это намного убедительнее, чем разница средних.

In [ ]:
def fold_diff(new_id, base_id="E7"):
    """MAE(new) - MAE(base) на каждом фолде, км. Минус = новая модель лучше."""
    return [mean_absolute_error(y_train[va], oofs[new_id][va]) - mean_absolute_error(y_train[va], oofs[base_id][va])
            for _, va in cv.split(train)]


cmp = pd.DataFrame({k: fold_diff(k) for k in ["E9a", "E9b", "E9c", "E9"]},
                   index=[f"фолд {i + 1}" for i in range(N_FOLDS)]).T.round(0)
cmp["среднее"] = cmp.mean(axis=1).round(0)
cmp["лучше E7 на фолдах"] = (cmp.iloc[:, :N_FOLDS] < 0).sum(axis=1).astype(str) + " из 5"
cmp

**Что здесь видно:**
- **флаг такси-модели (E9a)** улучшает ошибку на ≈ 140 км, и **на всех 5 фолдах**. Эффект маленький
  (0,3 %), но настоящий;
- **`возраст × такси` (E9b) ничего не добавляет** к E9a: наклон у такси почти такой же, как у всех.
  Гипотеза «такси набирают пробег быстрее» в этих данных не подтвердилась;
- **`возраст × газ` (E9c) — тоже ноль:** разница с E7 на фолдах в пределах ±12 км.

**Почему флаг помог, если модели и так есть в `brand_model`?** Ridge штрафует каждый вес отдельно
и «поджимает» вес модели, у которой мало машин (у Polo — 46). Общий флаг собирает все 7 моделей
в одну группу из ≈ 1 700 машин, и общий для них эффект Ridge почти не поджимает. Получается
«общий сдвиг для такси + небольшая поправка для конкретной модели».

### Где модель ошибается на такси

In [ ]:
seg9 = pd.DataFrame({
    "машин": [int(train.taxi_model.sum()), int(train.fuel_lpg.sum()), int((train.taxi_model == 0).sum())],
}, index=["такси-модели", "на газу", "не такси"])
for k in ["E7", "E9a"]:
    seg9[f"MAE {k}"] = [mean_absolute_error(y_train[m], oofs[k][m]) for m in
                        (train.taxi_model == 1, train.fuel_lpg == 1, train.taxi_model == 0)]
seg9["медиана пробега"] = [np.median(y_train[m]) for m in (train.taxi_model == 1, train.fuel_lpg == 1, train.taxi_model == 0)]
seg9.round(0)

**Что здесь видно:** на такси-моделях модель ошибается на ≈ 60 тыс. км — в полтора раза больше,
чем на остальных (≈ 41 тыс.), и флаг почти ничего не меняет. Причина в том, что **не каждая Sonata —
такси**. Среди них есть и такси с 400 тыс. км, и семейные машины со 100 тыс., а признаки у них
одинаковые. Чтобы их различать, нужен признак **конкретной машины**, а не модели: газ у этой
машины, слова в описании, цена относительно таких же Sonata того же года (эксперименты E10–E11).

### Тест для E9a — один раз

Выбираем самый простой вариант с тем же эффектом — **E9a** (E9b и E9 не лучше, а сложнее).

In [ ]:
final9 = build_model(E9a).fit(train, y_train)
m9, _ = test_metrics(final9)
r9 = pd.DataFrame(results).set_index("id").loc["E9a"]
pd.DataFrame({"CV MAE": [r_best["CV MAE"], r9["CV MAE"]], "test MAE": [m_test["MAE"], m9["MAE"]],
              "test R²": [m_test["R²"], m9["R²"]]}, index=["E7 (пилот)", "E9a"]).round({"CV MAE": 0, "test MAE": 0, "test R²": 3})

**Что здесь видно:** на тесте E9a и E7 почти одинаковы (разница ≈ 10 км). На CV выигрыш был
≈ 140 км — на 2 800 тестовых машинах такой маленький эффект просто тонет в шуме.

### Вывод по E9

1. **Флаг такси-модели — маленький плюс:** на CV лучше E7 на всех 5 фолдах (≈ −140 км), на тесте —
   без разницы. Признак простой и понятный, поэтому оставляем, но заметного улучшения он не дал.
2. **Взаимодействия с возрастом не помогли** — ни для такси, ни для газа. Не оставляем.
3. Главная проблема такси — **внутри** модели: одинаковые по признакам машины ездят очень
   по-разному. Дальше нужны признаки конкретной машины (E10 — цена относительно своей модели
   и года, E11 — слова из описания).

## 15. E10 — цена относительно таких же машин

**Идея.** Разбор машин с одинаковыми характеристиками показал: если машина дешевле таких же
(та же модель, тот же год), пробег у неё обычно больше. Покупатели знают пробег и закладывают
его в цену. Признак:

$$\text{rel\_price} = \log\frac{\text{цена машины}}{\text{медиана цены той же модели того же года}}$$

- `0` — цена как у всех таких машин; `−0,2` — на ≈ 18 % дешевле; `+0,2` — на ≈ 22 % дороже;
- если в обучении меньше 5 машин этой модели этого года — берём медиану модели по всем годам;
- нет цены («договорная») — ставим `0`, «как у всех» (пропуск цены модель и так видит через
  `MissingIndicator`).

**Утечка?** Цели здесь нет — мы усредняем **цену**, а не пробег. Но медианы — это тоже «обучение»,
поэтому честно считать их **только по обучающим машинам** каждого фолда, а test — по медианам train.

**Планка — заранее, до запуска:** E10 берём, если он лучше E9a **на всех 5 фолдах и в среднем
хотя бы на 500 км**.

### 🆕 Свой трансформер

Чтобы медианы считались внутри CV автоматически, делаем свой шаг для `Pipeline`. Любой шаг
sklearn — это класс с двумя методами:
- `fit(X)` — **запомнить** что-то по обучающим данным (у `StandardScaler` — среднее и SD,
  у нас — медианы цены);
- `transform(X)` — **применить** запомненное к любым данным (train, validation, test).

`Pipeline` сам вызывает `fit` только на обучающей части фолда — так и получается «честно».

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin


class RelativePrice(BaseEstimator, TransformerMixin):
    """log(цена / медиана цены той же модели того же года); clip — обрезка, см. ниже."""

    def __init__(self, min_count=5, clip=None):
        self.min_count = min_count
        self.clip = clip

    def fit(self, X, y=None):
        p = pd.DataFrame(X, columns=REL_COLS).dropna(subset=["price_kgs"])
        by_year = p.groupby(["brand_model", "year"]).price_kgs.agg(["median", "size"])
        by_model = p.groupby("brand_model").price_kgs.agg(["median", "size"])
        self.by_year_ = by_year.loc[by_year["size"] >= self.min_count, "median"]   # модель + год
        self.by_model_ = by_model.loc[by_model["size"] >= self.min_count, "median"]  # модель
        self.overall_ = p.price_kgs.median()                                         # все машины
        return self

    def transform(self, X):
        X = pd.DataFrame(X, columns=REL_COLS)
        keys = pd.MultiIndex.from_frame(X[["brand_model", "year"]])
        ref = pd.Series(keys.map(self.by_year_.to_dict().get), index=X.index, dtype=float)
        ref = ref.fillna(X.brand_model.map(self.by_model_)).fillna(self.overall_)
        rel = np.log(X.price_kgs.astype(float) / ref)
        if self.clip is not None:
            rel = rel.clip(-self.clip, self.clip)
        return rel.fillna(0.0).to_frame().to_numpy()

    def get_feature_names_out(self, input_features=None):
        return np.array(["rel_price"])


REL_COLS = ["brand_model", "year", "price_kgs"]
build_model_pilot = build_model   # функция из раздела 5 — не меняем


def build_model(exp):
    """Как раньше; если в эксперименте rel_price=True — добавляем шаг RelativePrice в ColumnTransformer."""
    model = build_model_pilot(exp)
    if exp.get("rel_price"):
        prep = model.named_steps["prep"]
        rel_step = Pipeline([("rel", RelativePrice(clip=exp.get("clip"))), ("scale", StandardScaler())])
        prep.set_params(transformers=prep.transformers + [("relp", rel_step, REL_COLS)])
    return model

Посмотрим на нескольких Sonata 2019 года на газу, что считает трансформер:

In [ ]:
demo_rp = RelativePrice().fit(train[REL_COLS])
son = train[(train.brand_model == "Hyundai | Sonata") & (train.year == 2019) & (train.fuel_lpg == 1)]
son = son.dropna(subset=["price_kgs"]).sort_values("price_kgs").iloc[[0, 10, 40, 80, -1]]
pd.DataFrame({"цена, сом": son.price_kgs,
              "медиана Sonata 2019": demo_rp.by_year_[("Hyundai | Sonata", 2019)],
              "rel_price": demo_rp.transform(son[REL_COLS])[:, 0].round(2),
              "пробег": son[TARGET]})

**Что здесь видно:**
- у машины «по рынку» `rel_price` около 0, дешевле рынка — меньше 0, дороже — больше 0;
- на пяти машинах связь с пробегом не видна — она проявляется только **в среднем**, на тысячах машин;
- самая «дешёвая» Sonata за 50 000 сом (≈ 570 $) — это не дешёвая машина, а **ошибка в цене**:
  `rel_price` = −3. Такие точки ещё помешают нам ниже.

In [ ]:
E10_raw = run(dict(E9a, id="E10", title="E9a + цена относительно модели и года", rel_price=True))
print("разница с E9a по фолдам:", [round(x) for x in fold_diff("E10", "E9a")])

**Что здесь видно:** всего ≈ −40 км. Признак сильный, а модели почти не помог. Разберёмся почему.

### Почему так мало

**Причина 1: модель уже почти знала это.** В ней есть `log_price`, марка×модель и возраст.
`log(цена / медиана)` = `log(цена) − log(медиана)`, а `log(медиана)` в основном определяется моделью
и годом. Линейная модель может «сложить» это сама. Проверим: насколько `rel_price` предсказывается
признаками, которые уже есть.

In [ ]:
rp_full = RelativePrice().fit(train[REL_COLS])
rel_train = rp_full.transform(train[REL_COLS])[:, 0]
X_e9a = build_model_pilot(E9a).named_steps["prep"].fit(train).transform(train)
r2_known = LinearRegression().fit(X_e9a, rel_train).score(X_e9a, rel_train)
print(f"R² «rel_price по признакам E9a» = {r2_known:.2f} -> {r2_known:.0%} информации модель уже имела")
print(f"rel_price: min {rel_train.min():.2f}, max {rel_train.max():.2f}  (log: −4,4 = в 80 раз дешевле медианы!)")

resid9 = y_train - oofs["E9a"]
q5 = pd.qcut(rel_train, 5, labels=["20 % самых дешёвых", "дешевле", "средне", "дороже", "20 % самых дорогих"])
pd.Series(resid9).groupby(q5, observed=True).mean().round(0).to_frame("средний остаток E9a, км")

**Причина 2: выбросы.** У `rel_price` есть машины «в 80 раз дешевле медианы» — это ошибки в цене
(сомы вместо долларов, цена за запчасти и т. п.), а не дешёвые машины. Посмотрите на таблицу:
у «20 % самых дешёвых» остаток **меньше**, чем у просто «дешевле» — тренд ломается как раз там,
где сидят выбросы. Линейная модель подгоняет одну прямую под всё, и эти точки «положили» её.

### 🆕 Обрезка выбросов (clip)

**Одной фразой:** всё, что дешевле −порога, считаем равным −порогу; всё, что дороже +порога, — +порогу.
Машина «в 80 раз дешевле» станет просто «дешевле на порог», и перестанет тянуть вес.
Порог — гиперпараметр, подбираем его по CV, как `alpha`.

In [ ]:
clip_rows = []
for c in [0.7, 0.5, 0.3, 0.2, 0.1]:
    eid = f"E10 clip {c}"
    run(dict(E9a, id=eid, title=f"E10, обрезка ±{c} (≈ ±{(np.exp(c) - 1):.0%})", rel_price=True, clip=c))
    diff = fold_diff(eid, "E9a")
    clip_rows.append({"порог": c, "≈ в %": f"±{(np.exp(c) - 1):.0%}", "обрезано машин": f"{(np.abs(rel_train) >= c).mean():.0%}",
                      "среднее vs E9a, км": round(np.mean(diff)), "лучше E9a на фолдах": f"{sum(x < 0 for x in diff)} из 5"})
pd.DataFrame(clip_rows)

**Что здесь видно:**
- обрезка сильно помогает: с −40 км до ≈ −300 км, и **на всех 5 фолдах**;
- после порога ±0,2 (≈ ±20 %) выигрыш почти не растёт. Берём **0,2**: он проще объясняется
  («дешевле/дороже рынка на 20 % и больше»), а более узкие пороги дают лишь несколько км;
- при таком пороге модели важно скорее **направление** (дешевле или дороже рынка), а не насколько.

### Решение по планке

In [ ]:
best10 = "E10 clip 0.2"
diff10 = fold_diff(best10, "E9a")
gain = -np.mean(diff10)
ok = all(x < 0 for x in diff10) and gain >= 500
print(f"{best10}: лучше E9a на {gain:.0f} км, на {sum(x < 0 for x in diff10)} из 5 фолдов")
print(f"планка (5 из 5 и ≥ 500 км): {'✅ пройдена' if ok else '❌ не пройдена'}")

**Что здесь видно:** E10 — **лучший признак из всех, что мы пробовали после пилота** (≈ −320 км,
5 из 5 фолдов), но заранее заданную планку в 500 км он **не прошёл**. По правилам, которые мы сами
задали до запуска, test для него не считаем — иначе планка теряет смысл.

### Вывод по E10

1. **Цена относительно рынка работает,** но слабее, чем казалось по разбору групп: ≈ 60 % этой
   информации модель уже получала из цены, модели и возраста.
2. **Выбросы в цене мешали:** без обрезки −40 км, с обрезкой ±20 % — −320 км.
3. **Планку 500 км E10 не прошёл.** Варианты: (а) принять как есть — признак устойчивый и понятный,
   но тогда честно записать, что планку мы опустили; (б) оставить на потом и объединить с
   E11 (слова из описания) — вместе они могут пройти планку.
4. Общая картина подтверждается: у машин с одинаковыми характеристиками пробег различается
   на ±50 %, и признаки из объявлений объясняют лишь малую часть этой разницы.

## 16. E8 — убираем заглушки в пробеге

**Идея.** Часть пробегов — явно не показания одометра: 55 555, 99 999, 123 456, 222 222. Продавец
не знает пробег (или не хочет писать) и ставит «что-нибудь». Если учить модель на таких машинах,
она учится на мусоре.

**Что считаем заглушкой** (правило придумано до запуска, по виду числа, а не по ошибке модели):
- одна цифра повторяется: 11 111, 55 555, 222 222, 777 777;
- цифры подряд: 12 345, 123 456, 234 567, 98 765;
- повтор блока: 123 123, 121 212;
- «почти круглое»: 199 999, 299 999, 499 999;
- мем 100 500.

In [ ]:
def is_placeholder(km):
    s = str(int(round(km)))
    same_digit = len(s) >= 4 and len(set(s)) == 1                        # 55555
    in_a_row = len(s) >= 5 and (s in "1234567890" or s in "9876543210")   # 123456, 98765
    block = len(s) == 6 and (s[:3] == s[3:] or s[:2] * 3 == s)            # 123123, 121212
    nines = len(s) == 6 and s[1:] == "99999"                              # 199999
    return same_digit or in_a_row or block or nines or s == "100500"


ph_train = train[TARGET].map(is_placeholder).to_numpy()
ph_test = test[TARGET].map(is_placeholder).to_numpy()
print(f"заглушек: train {ph_train.sum()} ({ph_train.mean():.1%}), test {ph_test.sum()}")
print(f"MAE E9a (OOF) на заглушках {sp(mean_absolute_error(y_train[ph_train], oofs['E9a'][ph_train]))} км, "
      f"на остальных {sp(mean_absolute_error(y_train[~ph_train], oofs['E9a'][~ph_train]))} км")
train.loc[ph_train, ["brand_model", "year", TARGET]].assign(прогноз_E9a=oofs["E9a"][ph_train].round(-3)).sample(10, random_state=0)

**Что здесь видно:** заглушек мало — ≈ 0,5 % машин. Модель на них ошибается почти вдвое сильнее
(≈ 80 тыс. км против ≈ 44 тыс.). Некоторые «заглушки» правдоподобны (Camry 2017 с 199 999 км —
возможно, это правда «почти 200 тысяч»), а некоторые — явный мусор (Polo 2017 с 55 555 км).

### Как сравнивать честно

Здесь меняется не модель, а **данные для обучения**. Поэтому:
- в каждом фолде **учим** E8 на train-части **без заглушек**, а E9a — как раньше, на всех;
- **проверяем** обе модели на **одних и тех же** машинах validation-части — в двух вариантах:
  на всех машинах и только на «чистых» (без заглушек).

Если заглушки выкинуть и из проверки, ошибка станет меньше просто потому, что из проверки ушли
самые трудные машины. Это не улучшение модели — поэтому обе модели всегда меряем на одном наборе.

In [ ]:
def cv_without(exp, drop_mask):
    """5-fold CV: учим без строк drop_mask, проверяем на всех машинах фолда и на «чистых»."""
    on_all, on_clean = [], []
    for tr_idx, va_idx in cv.split(train):
        fit_idx = tr_idx[~drop_mask[tr_idx]]
        model = build_model(exp).fit(train.iloc[fit_idx], y_train[fit_idx])
        p = model.predict(train.iloc[va_idx])
        ok = ~ph_train[va_idx]
        on_all.append(mean_absolute_error(y_train[va_idx], p))
        on_clean.append(mean_absolute_error(y_train[va_idx][ok], p[ok]))
    return np.array(on_all), np.array(on_clean)


nothing = np.zeros(len(train), bool)
e9a_all, e9a_clean = cv_without(E9a, nothing)        # E9a: учим на всех
e8_all, e8_clean = cv_without(E9a, ph_train)          # E8: учим без заглушек

pd.DataFrame({
    "E9a (учим на всех)": [e9a_all.mean(), e9a_clean.mean()],
    "E8 (учим без заглушек)": [e8_all.mean(), e8_clean.mean()],
    "разница по фолдам": [[round(x) for x in e8_all - e9a_all], [round(x) for x in e8_clean - e9a_clean]],
    "лучше на фолдах": [f"{(e8_all < e9a_all).sum()} из 5", f"{(e8_clean < e9a_clean).sum()} из 5"],
}, index=["проверка на всех машинах", "проверка на «чистых»"]).round(0)

**Что здесь видно:** **заглушки не мешают** — без них модель не стала лучше ни на всех машинах,
ни на «чистых». Разница по фолдам скачет в обе стороны (от −60 до +280 км) — это шум.

**Почему так:**
1. Заглушек ≈ 50 на 11 тыс. машин. На оценку ≈ 170 весов они почти не влияют.
2. Многие «заглушки» близки к реальному пробегу: 199 999, 222 222, 123 456 — это не 0 и не миллион.
   По ошибке модели видно: прогноз для них часто в тех же пределах, что и для обычных машин.

**Решение:** E8 не берём, test не считаем. Правило `is_placeholder` пригодится в отчёте —
ошибку на «чистых» машинах можно указывать рядом с обычной.

### Где тогда шум в цели

Отдельный разбор машин с одинаковыми характеристиками (модель, год, топливо, двигатель, коробка,
кузов, привод) показал более крупные источники:
- **круглый пробег** (кратен 10 000) — **27 %** машин; у них пробег на ≈ 15 % выше, чем у таких же
  машин с «точным» пробегом — похоже на оценку «на глаз»;
- **пробег, исправленный правилом ×1000** (продавец написал «250» вместо «250 000») — 8 %.

Выкидывать их нельзя — это четверть данных и обычная манера продавцов. Но именно они, а не
заглушки, задают «шумовой пол» ошибки модели.

---
# 17. Итоги по всем экспериментам

Одна таблица на все эксперименты ноутбука. Таблица собирается из результатов выше, поэтому
при перезапуске она обновится сама.

- **CV MAE** — средняя ошибка на 5 фолдах train, км; **±** — разброс между фолдами;
- **Δ к базе** — насколько эксперимент лучше (минус) или хуже (плюс) своей базы, км;
- **лучше базы на фолдах** — на скольких из 5 фолдов эксперимент лучше базы;
- **test MAE** — только у моделей, которые прошли отбор (test считаем один раз).

In [ ]:
res_all = pd.DataFrame(results).set_index("id")

# база = с чем сравнивали эксперимент, когда принимали решение
BASE = {"E1": "B2", "E2": "E1", "E3": "E2", "E4": "E2", "E5": "E2", "E6": "E5", "E7": "E6",
        "E9a": "E7", "E9b": "E9a", "E9c": "E7", "E9": "E9a",
        "E10": "E9a", "E10 clip 0.7": "E9a", "E10 clip 0.5": "E9a", "E10 clip 0.3": "E9a",
        "E10 clip 0.2": "E9a", "E10 clip 0.1": "E9a"}
DECISION = {
    "B0": "бейзлайн", "B1": "бейзлайн", "B2": "бейзлайн — планка пилота",
    "E1": "✅ принят", "E2": "✅ принят (log цены удобнее читать)", "E3": "❌ хуже E2", "E4": "❌ хуже E2",
    "E5": "✅ принят (Ridge нужен для E6)", "E6": "✅ принят", "E7": "✅ модель пилота",
    "E9a": "✅ принят (маленький плюс)", "E9b": "❌ ничего не добавил", "E9c": "❌ ничего не добавил",
    "E9": "❌ не лучше E9a",
    "E10": "❌ выбросы цены «гасят» признак", "E10 clip 0.7": "подбор порога", "E10 clip 0.5": "подбор порога",
    "E10 clip 0.3": "подбор порога", "E10 clip 0.2": "⚠️ лучший, но ниже планки 500 км",
    "E10 clip 0.1": "подбор порога (почти как 0.2)",
    "E8": "❌ заглушки не мешают",
}
TEST = {"B0": table.loc["B0 (среднее)", "MAE"], "B2": table.loc["B2 (возраст)", "MAE"],
        "E7": m_test["MAE"], "E9a": m9["MAE"]}

rows = []
for eid, r in res_all.iterrows():
    base = BASE.get(eid)
    diff = fold_diff(eid, base) if base else None
    rows.append({"id": eid, "что меняли": r["что меняем"], "CV MAE": r["CV MAE"], "±": r["± (std по фолдам)"],
                 "база": base or "", "Δ к базе": np.mean(diff) if diff else np.nan,
                 "лучше базы на фолдах": f"{sum(x < 0 for x in diff)} из 5" if diff else "",
                 "test MAE": TEST.get(eid, np.nan), "решение": DECISION.get(eid, "")})

# E8 считался отдельной функцией (учим без заглушек) — добавляем вручную
rows.append({"id": "E8", "что меняли": "E9a, но учим без заглушек пробега", "CV MAE": e8_all.mean(),
             "±": e8_all.std(), "база": "E9a", "Δ к базе": (e8_all - e9a_all).mean(),
             "лучше базы на фолдах": f"{(e8_all < e9a_all).sum()} из 5", "test MAE": np.nan, "решение": DECISION["E8"]})

summary = pd.DataFrame(rows).set_index("id")
summary.style.format({"CV MAE": "{:,.0f}", "±": "{:,.0f}", "Δ к базе": "{:+,.0f}", "test MAE": "{:,.0f}"},
                     na_rep="—").set_properties(subset=["что меняли", "решение"], **{"text-align": "left"})

In [ ]:
# путь лучшей модели: каждый принятый шаг
path = ["B2", "E1", "E2", "E5", "E6", "E7", "E9a", "E10 clip 0.2"]
p = summary.loc[path]
colors = [GREY if i.startswith("B") else (ORANGE if i == "E10 clip 0.2" else BLUE) for i in p.index]

fig, ax = plt.subplots(figsize=(9, 3.8))
ax.bar(range(len(p)), p["CV MAE"], yerr=p["±"], color=colors, width=0.6, error_kw=dict(lw=1, capsize=3))
for i, v in enumerate(p["CV MAE"]):
    ax.text(i, v + 600, f"{v / 1000:.1f}k", ha="center", fontsize=8.5)
ax.set_xticks(range(len(p)), [i.replace(" clip ", "\nclip ") for i in p.index], fontsize=8.5)
ax.set_ylim(40_000, p["CV MAE"].max() * 1.04)
ax.yaxis.set_major_formatter(mt.FuncFormatter(km))
ax.grid(axis="x", visible=False)
ax.set_title("Как менялась ошибка (CV MAE) по принятым шагам; оранжевый — E10, ниже планки")
plt.show()

total = summary.loc["B2", "CV MAE"] - summary.loc["E9a", "CV MAE"]
print(f"от B2 до E9a: −{sp(total)} км; из них пилот (B2 → E7): −{sp(summary.loc['B2', 'CV MAE'] - summary.loc['E7', 'CV MAE'])} км, "
      f"итерация 2 (E7 → E9a): −{sp(summary.loc['E7', 'CV MAE'] - summary.loc['E9a', 'CV MAE'])} км")

### Что получилось

**Пилот (B0 → E7).** Основной выигрыш дали два шага: все базовые признаки (E1, ≈ −6,8 тыс. км к B2)
и марка×модель (E6, ≈ −1,2 тыс. км). Логарифм цели и «пробег в год» не помогли. Модель пилота E7:
CV 44,1 тыс. км, test 45,0 тыс. км — обе планки пилота пройдены.

**Итерация 2 (после E7).**

| эксперимент | что проверяли | итог |
|---|---|---|
| E9a–E9 | такси-модели, возраст × такси, возраст × газ | флаг такси — маленький, но устойчивый плюс; взаимодействия — ноль |
| E10 | цена относительно той же модели и года | работает только с обрезкой выбросов цены; −316 км — лучший шаг итерации, но ниже планки 500 км |
| E8 | обучение без заглушек пробега | заглушек 0,5 %, на модель не влияют |

**Главный вывод.** После пилота все эксперименты вместе дали меньше 0,5 тыс. км. Это не потому,
что идеи плохие: у машин с одинаковыми характеристиками пробег различается на ±50 %, и признаки
из объявлений объясняют лишь малую часть этой разницы. Модель упирается в **данные**, а не в метод.

### Открытые вопросы
1. **E10** — принять с пониженной планкой или проверить вместе с E11 (слова из описания).
2. **Цены** — починить в `prep.py` рассрочку, аренду, «$ как сомы» и лишние нули (≈ 1–2 % машин),
   затем перезапустить E10 без обрезки.
3. **Повторы одной машины** — без фото мы находим 402 повтора вместо 2 252; оценка может быть
   немного оптимистичной.
4. **Постановка** — цель MAE ≤ 35 тыс. км, скорее всего, недостижима на этих данных. Полезнее
   предсказывать **интервал** пробега или искать **подозрительно маленький** заявленный пробег.